# 🪟 Python Monotonic Deque — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Imagine a sliding window moving across a row of numbers, and you need the maximum inside at all times.
> Instead of re-scanning the whole window each step, keep a waiting line (deque) that auto-evicts
> anything that can never be the maximum — because something larger already sits behind it.
> The front of the line is always the current maximum. Old entries leave from the front when the window moves past them.

---

## 📋 Table of Contents

| # | Section |
|---|------|
| 1 | [What Is a Monotonic Deque? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Sliding Window Maximum (LC 239)](#5) |
| 6 | [Pattern 2: Longest Subarray Abs Diff ≤ Limit (LC 1438)](#6) |
| 7 | [Pattern 3: Shortest Subarray with Sum ≥ K (LC 862)](#7) |
| 8 | [The Monotonic Deque Decision Map](#8) |
| 9 | [Interview Cheat Sheet](#9) |
| 10 | [Summary Map](#10) |

<a id='1'></a>
## 1. What Is a Monotonic Deque? The Visual Model

---

```
SLIDING WINDOW MAXIMUM — nums=[1,3,-1,-3,5,3,6,7], k=3

  Window [1,3,-1]: deque stores indices of USEFUL values only

    front                        back
      │                            │
      ▼                            ▼
    ┌─────┬─────┐              ┌─────┬─────┐
    │ idx │val  │              │ idx │val  │
    │  1  │  3  │    →→→       │  2  │ -1  │
    └─────┴─────┘              └─────┴─────┘
    maximum                    most recent

  When a new element arrives:
    LEFT  eviction: front index is out of window → popleft()
    RIGHT eviction: back value < new value → pop() (it can never be max)

  Result: front always holds the index of the current window maximum.

WHY DOES THIS MATTER?
  Brute force sliding max: O(n·k) — scan k elements each step
  Monotonic deque:         O(n)   — each element appended once, removed once
```

<a id='2'></a>
## 2. Creating / Setup

In [ ]:
from collections import deque

dq = deque()                            # empty deque — the only import needed
print("empty deque:", dq)

dq_from_list = deque([3, 1, 4, 1, 5])  # preloaded with values
print("from list:", dq_from_list)

# Monotonic deque stores INDICES (not values) — allows window-expiry check
nums = [1, 3, -1, -3, 5]
mono_dq = deque()                       # will store indices; decreasing by value
for i, x in enumerate(nums):
    while mono_dq and nums[mono_dq[-1]] <= x:   # evict useless back entries
        mono_dq.pop()
    mono_dq.append(i)
print("decreasing index-deque for", nums, ":", list(mono_dq))
print("current max index:", mono_dq[0], "value:", nums[mono_dq[0]])

# Two-deque setup for tracking both max and min (LC 1438)
max_dq = deque()                        # front = index of current max
min_dq = deque()                        # front = index of current min
print("max_dq and min_dq initialized.")

<a id='3'></a>
## 3. The Core API — All Operations

```
OPERATION              COMPLEXITY   WHAT IT DOES
────────────────────────────────────────────────────────────
dq.append(x)           O(1)         push to RIGHT (back) — new arrivals
dq.appendleft(x)       O(1)         push to LEFT (front) — rare in mono pattern
dq.pop()               O(1)         remove from RIGHT — evict stale back entries
dq.popleft()           O(1)         remove from LEFT — evict expired window entries
dq[0]                  O(1)         peek front — current window max (or min)
dq[-1]                 O(1)         peek back — most recently added
not dq                 O(1)         empty check
────────────────────────────────────────────────────────────
AMORTIZED GUARANTEE: each element appended once and removed at most once
→ total operations across n steps = O(n)

TWO EVICTION PATTERNS (used together for sliding window max):
  RIGHT eviction (maintain ordering):  while dq and nums[dq[-1]] < x: dq.pop()
  LEFT eviction  (window expiry):      if dq and dq[0] < left_bound: dq.popleft()

THINGS YOU DO NOT DO
────────────────────
❌  Store values instead of indices — you can't check window expiry without index
❌  Use dq[1], dq[2] — only dq[0] (max) and dq[-1] (back) matter
❌  Forget the LEFT eviction — stale indices outside the window corrupt the answer
❌  Use a regular list for this — deque gives O(1) popleft; list.pop(0) is O(n)
```

In [ ]:
from collections import deque

# Live demo: sliding window max with k=3 on [1,3,-1,-3,5,3,6,7]
nums = [1, 3, -1, -3, 5, 3, 6, 7]
k = 3
dq = deque()                            # stores indices, decreasing by value
result = []

print("Sliding window max demo (k=3):")
print(f"  {'i':>2}  {'x':>3}  {'right_evict':>14}  {'left_evict':>12}  {'dq':>14}  {'window_max':>10}")
print("  " + "-" * 60)

for i, x in enumerate(nums):
    right_ev = []
    while dq and nums[dq[-1]] < x:          # RIGHT: evict smaller back entries
        right_ev.append(nums[dq.pop()])
    dq.append(i)

    left_ev = None
    if dq[0] < i - k + 1:                   # LEFT: evict expired front entry
        left_ev = dq.popleft()

    win_max = nums[dq[0]] if i >= k - 1 else "—"
    if i >= k - 1:
        result.append(nums[dq[0]])
    print(f"  {i:>2}  {x:>3}  {str(right_ev):>14}  {str(left_ev):>12}  {str(list(dq)):>14}  {str(win_max):>10}")

print("\nResult:", result)  # [3,3,5,5,6,7]

<a id='4'></a>
## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                        WHAT TO DO
──────────────────────────────────────────────────────────────────────
"max (or min) in sliding window of size k"   Single decreasing deque of indices
"longest subarray where max-min <= limit"    Two deques: max_dq + min_dq, expand/shrink
"shortest subarray with sum >= K"            Prefix sum + increasing deque, look for diff >= K
Fixed window size                            One deque + left expiry at i - k
Variable window (shrink when invalid)        Two pointers: left pointer instead of fixed expiry
Need BOTH max and min of window              Two separate deques (one decreasing, one increasing)
──────────────────────────────────────────────────────────────────────
DEQUE TYPE:
  DECREASING deque → front = MAX of window
  INCREASING deque → front = MIN of window (or smallest prefix sum for LC 862)
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Sliding Window Maximum — LC 239

---

```
PROBLEM: Given nums and window size k, return the max of each sliding window.

TRICK: Monotonic DECREASING deque of indices.
       Before appending i: evict all back indices whose values are < nums[i].
       After appending i: evict front if it's outside the window (i - dq[0] >= k).
       The front always holds the max index of the current window.

SLOW MOTION TRACE on nums=[1,3,-1,-3,5,3,6,7], k=3:
  i=0 x=1   right_evict=[]     dq=[0]     (window not full yet)
  i=1 x=3   right_evict=[0→1]  dq=[1]     (window not full yet)
  i=2 x=-1  right_evict=[]     dq=[1,2]   max=nums[1]=3  ✓
  i=3 x=-3  right_evict=[]     dq=[1,2,3] left_evict: dq[0]=1, 1<3-3+1=1? no
                                            max=nums[1]=3  ✓
  i=4 x=5   right_evict=[2→-1,3→-3,1→3]  dq=[4]  max=nums[4]=5  ✓
  i=5 x=3   right_evict=[]     dq=[4,5]   max=nums[4]=5  ✓
  i=6 x=6   right_evict=[5→3,4→5]  dq=[6] max=nums[6]=6  ✓
  i=7 x=7   right_evict=[6→6]  dq=[7]     max=nums[7]=7  ✓
  result = [3, 3, 5, 5, 6, 7]

KEY INSIGHT: Two types of eviction — one for ordering (RIGHT), one for expiry (LEFT).

TIME:  O(n)  — each index appended once, removed at most once
SPACE: O(k)  — deque holds at most k indices at any time
```

In [ ]:
from collections import deque

def maxSlidingWindow(nums, k):
    """
    LC 239 — Sliding Window Maximum
    Approach: Monotonic decreasing deque of indices; front = max of current window.
    Args:
        nums (List[int]): input array.
        k (int): window size, 1 <= k <= len(nums).
    Returns:
        List[int]: max value of each window position.
    Time:  O(n)  — each index appended once and removed at most once
    Space: O(k)  — deque holds at most k indices simultaneously
    """
    dq = deque()                        # decreasing by value; stores indices
    result = []

    # Slow motion on [1,3,-1,-3,5,3,6,7], k=3:
    # i=0  append 0                dq=[0]
    # i=1  evict 0 (1<3)  append 1 dq=[1]
    # i=2  append 2                dq=[1,2]  → result[0]=nums[1]=3
    # i=3  append 3                dq=[1,2,3] → result[1]=3
    # i=4  evict 3,2,1  append 4   dq=[4]  → result[2]=5
    # i=5  append 5                dq=[4,5] → result[3]=5
    # i=6  evict 5,4  append 6     dq=[6]  → result[4]=6
    # i=7  evict 6  append 7       dq=[7]  → result[5]=7

    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] < x:  # RIGHT evict: back values smaller than x are useless
            dq.pop()
        dq.append(i)

        if dq[0] < i - k + 1:           # LEFT evict: front index fell outside the window
            dq.popleft()

        if i >= k - 1:                  # window is full — record the front (max index)
            result.append(nums[dq[0]])

    return result


def test_harness(fn):
    tests = [
        ([1, 3, -1, -3, 5, 3, 6, 7], 3, [3, 3, 5, 5, 6, 7]),
        ([1], 1, [1]),                                          # single element
        ([9, 11], 2, [11]),                                     # window = full array
        ([4, 3, 2, 1], 2, [4, 3, 2]),                          # strictly decreasing
        ([1, 2, 3, 4], 2, [2, 3, 4]),                          # strictly increasing
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


print(maxSlidingWindow([1, 3, -1, -3, 5, 3, 6, 7], 3))  # [3,3,5,5,6,7]
print(maxSlidingWindow([4, 3, 2, 1], 2))                  # [4,3,2]
print(maxSlidingWindow([1, 2, 3, 4], 2))                  # [2,3,4]
test_harness(maxSlidingWindow)
print("maxSlidingWindow defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Longest Subarray with Abs Diff ≤ Limit — LC 1438

---

```
PROBLEM: Find the longest subarray where |max - min| <= limit.

TRICK: Two simultaneous deques — one tracks window max (decreasing), one tracks
       window min (increasing). Shrink left when max - min > limit.
       This avoids re-scanning the window after each shrink.

SLOW MOTION TRACE on nums=[8,2,4,7], limit=4:
  left=0
  i=0 x=8   max_dq=[0]  min_dq=[0]   max=8  min=8  diff=0<=4  len=1
  i=1 x=2   max_dq=[0]  min_dq=[1]   max=8  min=2  diff=6>4
             SHRINK: left=1, evict 0 from both if == left-1
             max_dq=[0] dq[0]=0 < left=1 → popleft → max_dq=[]
             now max_dq=[] so max=nums[1]=2, min_dq=[1] min=2 diff=0  len=1
  i=2 x=4   max_dq=[2]  min_dq=[1,2]  max=4  min=2  diff=2<=4  len=2
  i=3 x=7   max_dq=[3]  min_dq=[1,2,3] max=7 min=2  diff=5>4
             SHRINK: left=2, evict stale
             min_dq[0]=1 < 2 → popleft → min_dq=[2,3]  min=4  diff=3<=4  len=2
  best = 2

KEY INSIGHT: Track max AND min simultaneously; shrink only when violated.
             Two deques replace O(n²) brute force with O(n).

TIME:  O(n)  — each index pushed and popped from each deque at most once
SPACE: O(n)  — two deques, each at most n entries
```

In [ ]:
from collections import deque

def longestSubarray(nums, limit):
    """
    LC 1438 — Longest Continuous Subarray with Abs Diff <= Limit
    Approach: Two deques track window max and min; shrink left when max-min > limit.
    Args:
        nums (List[int]): input array.
        limit (int): max allowed difference between window max and min.
    Returns:
        int: length of longest valid subarray.
    Time:  O(n)  — each index processed once in each deque
    Space: O(n)  — both deques bounded by n
    """
    max_dq = deque()                    # decreasing: front = index of window max
    min_dq = deque()                    # increasing: front = index of window min
    left = 0
    best = 0

    # Slow motion on [8,2,4,7], limit=4:
    # i=0  max_dq=[0] min_dq=[0]  diff=0  best=1
    # i=1  max_dq=[0] min_dq=[1]  diff=6>4 → shrink: left=1, pop 0 from max_dq  best=1
    # i=2  max_dq=[2] min_dq=[1,2] diff=2  best=2
    # i=3  max_dq=[3] min_dq=[1,2,3] diff=5>4 → shrink: left=2, pop 1 from min_dq  best=2

    for i, x in enumerate(nums):
        while max_dq and nums[max_dq[-1]] <= x:   # maintain decreasing max deque
            max_dq.pop()
        max_dq.append(i)

        while min_dq and nums[min_dq[-1]] >= x:   # maintain increasing min deque
            min_dq.pop()
        min_dq.append(i)

        while nums[max_dq[0]] - nums[min_dq[0]] > limit:  # window violated
            left += 1                                       # shrink from the left
            if max_dq[0] < left:                           # max leader expired
                max_dq.popleft()
            if min_dq[0] < left:                           # min leader expired
                min_dq.popleft()

        best = max(best, i - left + 1)

    return best


def test_harness(fn):
    tests = [
        ([8, 2, 4, 7], 4, 2),                      # example from LC
        ([10, 1, 2, 4, 7, 2], 5, 4),               # [2,4,7,2]? no [1,2,4] len=3; [2,4,7]? diff=5 ok → 4
        ([4, 2, 2, 2, 4, 4, 2, 2], 0, 3),          # only same values count
        ([1, 5, 6, 7, 8, 10, 6, 5, 6], 4, 5),
        ([1], 0, 1),                                # single element
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


print(longestSubarray([8, 2, 4, 7], 4))              # 2
print(longestSubarray([4, 2, 2, 2, 4, 4, 2, 2], 0)) # 3
print(longestSubarray([1], 0))                        # 1
test_harness(longestSubarray)
print("longestSubarray defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Shortest Subarray with Sum ≥ K — LC 862

---

```
PROBLEM: Find the shortest subarray with sum >= K. Array may contain negatives.

TRICK: Prefix sum + monotonic INCREASING deque.
       prefix[j] - prefix[i] = sum of nums[i..j-1]
       For each j, we want the LARGEST i where prefix[i] <= prefix[j] - K.
       A deque of prefix sum indices in increasing order lets us greedily pop from
       the front when the condition is met — giving the LONGEST left span.
       Once popped, that index is useless for all future j (shorter subarrays only).

SLOW MOTION TRACE on nums=[2,-1,2], K=3:
  prefix = [0, 2, 1, 3]
  dq = deque()  (stores indices into prefix array)

  j=0  prefix[0]=0   dq=[]       append 0     dq=[0]
  j=1  prefix[1]=2   dq=[0]      append 1     dq=[0,1]
  j=2  prefix[2]=1   1<=2: pop 1 (2>1 so increasing violated)  dq=[0]
                     append 2    dq=[0,2]
  j=3  prefix[3]=3   check front: prefix[3]-prefix[0]=3>=K=3 → len=3-0=3  pop 0
                     dq=[2]  prefix[3]-prefix[2]=2<3, stop
                     append 3    dq=[2,3]
  shortest = 3

KEY INSIGHT: Prefix sum converts "subarray sum" to a difference problem.
             Increasing deque on prefix values enables O(n) left-boundary search.

TIME:  O(n)  — one pass; each index appended and popped at most once
SPACE: O(n)  — prefix array + deque
```

In [ ]:
from collections import deque

def shortestSubarray(nums, k):
    """
    LC 862 — Shortest Subarray with Sum at Least K
    Approach: Prefix sum + monotonic increasing deque of prefix indices.
    Args:
        nums (List[int]): may contain negatives.
        k (int): target sum threshold.
    Returns:
        int: length of shortest subarray with sum >= k, or -1 if none.
    Time:  O(n)  — one pass through prefix array; each index pushed/popped once
    Space: O(n)  — prefix array of size n+1, deque of at most n+1 indices
    """
    n = len(nums)
    prefix = [0] * (n + 1)             # prefix[i] = sum of nums[0..i-1]
    for i in range(n):
        prefix[i + 1] = prefix[i] + nums[i]

    # Slow motion on [2,-1,2], k=3, prefix=[0,2,1,3]:
    # j=0  dq=[0]
    # j=1  prefix[1]=2>=prefix[0]=0, append  dq=[0,1]
    # j=2  prefix[2]=1 < prefix[1]=2 → pop 1 (maintain increasing) → dq=[0]
    #       1>=0, append  dq=[0,2]
    # j=3  prefix[3]=3, front=0: 3-0=3>=k → best=3-0=3, pop 0  dq=[2]
    #       front=2: 3-1=2<k, stop.  append 3  dq=[2,3]
    # result = 3

    dq = deque()                        # increasing by prefix value; stores indices
    best = float("inf")

    for j in range(n + 1):             # j is the right boundary (exclusive) index
        while dq and prefix[j] - prefix[dq[0]] >= k:   # valid subarray found
            best = min(best, j - dq.popleft())           # pop: this i won't be better for larger j

        while dq and prefix[dq[-1]] >= prefix[j]:       # maintain increasing order
            dq.pop()                    # back prefix >= current: back can never give smaller subarray
        dq.append(j)

    return best if best != float("inf") else -1


def test_harness(fn):
    tests = [
        ([2, -1, 2], 3, 3),             # spans whole array
        ([1], 1, 1),                    # single element equals k
        ([1], 2, -1),                   # impossible
        ([2, 1, -5, 4, 3], 4, 2),      # [4,3]? sum=7>=4, len=2; yes
        ([-1, -1, 1, 1, 1, 2], 3, 3),  # [1,1,1]
        ([84, -37, 32, 40, 95], 167, 3), # [32,40,95]
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


print(shortestSubarray([2, -1, 2], 3))          # 3
print(shortestSubarray([2, 1, -5, 4, 3], 4))    # 2
print(shortestSubarray([1], 2))                  # -1
test_harness(shortestSubarray)
print("shortestSubarray defined.")

<a id='8'></a>
## 8. The Monotonic Deque Decision Map

```
QUESTION TYPE                                KEY TECHNIQUE                    LC
─────────────────────────────────────────────────────────────────────────────────
Max of every fixed-size window               Decreasing deque of indices       239
Min of every fixed-size window               Increasing deque of indices       —
Longest subarray where max-min <= limit      Two deques (max+min), shrink left 1438
Shortest subarray with sum >= K (negatives)  Prefix sum + increasing deque     862

DEQUE TYPE GUIDE:
  Decreasing (back evict when nums[dq[-1]] < x) → front = WINDOW MAX
  Increasing (back evict when nums[dq[-1]] > x) → front = WINDOW MIN
  Increasing on PREFIX SUM                       → front = smallest prefix (earliest valid left)

TWO EVICTION TYPES (both needed for fixed window):
  RIGHT eviction: maintain ordering invariant  → dq.pop()
  LEFT  eviction: remove expired window entry  → dq.popleft()

VS MONOTONIC STACK:
  Stack: no left boundary — answers reference past elements arbitrarily
  Deque: left boundary enforced — answers confined to a sliding window
  Use Stack for NGE/histogram. Use Deque when window size or range is bounded.
```

<a id='9'></a>
## 9. Interview Cheat Sheet

**When to reach for a Monotonic Deque:**

| Signal | What to Do |
|--------|------------|
| "max (or min) of every window" | Decreasing deque of indices |
| "longest subarray, max-min <= limit" | Two deques, variable left pointer |
| "shortest subarray, sum >= K" (with negatives) | Prefix sum + increasing deque |
| Fixed window size k | Left expiry: `if dq[0] < i - k + 1: dq.popleft()` |
| Variable window | Shrink left with `while` loop when constraint violated |

**The O(1) operations — memorize these:**
```python
from collections import deque
dq = deque()       # init
dq.append(i)       # push to back (new element)
dq.pop()           # remove from back (ordering eviction)
dq.popleft()       # remove from front (expiry eviction)
dq[0]              # peek front = current max (or min)
dq[-1]             # peek back = most recent
not dq             # empty check
```

**Common templates:**
```python
# TEMPLATE 1: Sliding Window Maximum (fixed k)
from collections import deque
dq = deque()        # decreasing; stores indices
result = []
for i, x in enumerate(nums):
    while dq and nums[dq[-1]] < x:       # right eviction
        dq.pop()
    dq.append(i)
    if dq[0] < i - k + 1:               # left eviction (expiry)
        dq.popleft()
    if i >= k - 1:
        result.append(nums[dq[0]])

# TEMPLATE 2: Two-deque longest window (variable size)
from collections import deque
max_dq, min_dq = deque(), deque()
left = 0
best = 0
for i, x in enumerate(nums):
    while max_dq and nums[max_dq[-1]] <= x: max_dq.pop()
    max_dq.append(i)
    while min_dq and nums[min_dq[-1]] >= x: min_dq.pop()
    min_dq.append(i)
    while nums[max_dq[0]] - nums[min_dq[0]] > limit:
        left += 1
        if max_dq[0] < left: max_dq.popleft()
        if min_dq[0] < left: min_dq.popleft()
    best = max(best, i - left + 1)

# TEMPLATE 3: Shortest subarray sum >= K (with negatives)
from collections import deque
prefix = [0] * (n + 1)
for i in range(n): prefix[i+1] = prefix[i] + nums[i]
dq = deque()        # increasing by prefix value; stores indices
best = float("inf")
for j in range(n + 1):
    while dq and prefix[j] - prefix[dq[0]] >= k:
        best = min(best, j - dq.popleft())
    while dq and prefix[dq[-1]] >= prefix[j]:
        dq.pop()
    dq.append(j)
```

**Gotchas to not forget:**
```
❌  Storing values instead of indices — can't do expiry check without the index
❌  Using a list for popleft() — list.pop(0) is O(n), deque.popleft() is O(1)
❌  Forgetting the LEFT eviction for fixed windows — stale indices corrupt max
❌  Using one deque when you need both max and min (LC 1438)
✅  Amortized O(n): the total push+pop across all iterations = 2n
✅  For shortest subarray with negatives, prefix sum makes it a 1D problem
✅  Two deques track max AND min independently — shrink left only when both agree
```

<a id='10'></a>
## 10. Summary Map

```
                    MONOTONIC DEQUE
                          │
        ┌─────────────────┼─────────────────┐
        │                 │                 │
  SINGLE DEQUE       TWO DEQUES        DEQUE + PREFIX SUM
  (max or min)       (max AND min)      (sum problems)
        │                 │                 │
    LC 239            LC 1438           LC 862
  Sliding Max       Longest Window    Shortest Subarray
  fixed size k       max-min≤limit      sum≥K

EVICTION RULES SUMMARY:
  RIGHT (ordering):  while dq and nums[dq[-1]] OP x: dq.pop()
  LEFT  (expiry):    if dq[0] < left_bound: dq.popleft()

COMPLEXITY GUARANTEE:
  Each index → 1 append + at most 1 pop = O(1) amortized
  Total across n elements = O(n)

DEQUE VS STACK:
  Stack   → no window boundary → NGE, histogram
  Deque   → bounded window     → sliding max/min, subarray constraints
```

---
*End of Monotonic Deque Master Guide — Sean Edition*